In [0]:
# Definir el esquema con StructType (esquema mejor estructurado)
# Modulo y librerias
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType

# Definimos el esquema de las 31 columnas del dataset
schema = StructType([
    StructField("Time", DoubleType(), True),
    StructField("V1", DoubleType(), True),
    StructField("V2", DoubleType(), True),
    StructField("V3", DoubleType(), True),
    StructField("V4", DoubleType(), True),
    StructField("V5", DoubleType(), True),
    StructField("V6", DoubleType(), True),
    StructField("V7", DoubleType(), True),
    StructField("V8", DoubleType(), True),
    StructField("V9", DoubleType(), True),
    StructField("V10", DoubleType(), True),
    StructField("V11", DoubleType(), True),
    StructField("V12", DoubleType(), True),
    StructField("V13", DoubleType(), True),
    StructField("V14", DoubleType(), True),
    StructField("V15", DoubleType(), True),
    StructField("V16", DoubleType(), True),
    StructField("V17", DoubleType(), True),
    StructField("V18", DoubleType(), True),
    StructField("V19", DoubleType(), True),
    StructField("V20", DoubleType(), True),
    StructField("V21", DoubleType(), True),
    StructField("V22", DoubleType(), True),
    StructField("V23", DoubleType(), True),
    StructField("V24", DoubleType(), True),
    StructField("V25", DoubleType(), True),
    StructField("V26", DoubleType(), True),
    StructField("V27", DoubleType(), True),
    StructField("V28", DoubleType(), True),
    StructField("Amount", DoubleType(), True),
    StructField("Class", IntegerType(), True)
])

print("✅ Esquema definido correctamente")

In [0]:
# Leer el CSV con Try/Except
try:
    # Intentamos leer el archivo CSV con el esquema definido
    df = spark.read.format("csv") \
        .option("header", "true") \
        .schema(schema) \
        .load("/Volumes/fraud_detection/bronze/raw_data/input/csv/creditcard.csv")
    
    print(" Archivo leído correctamente")
    print(f" Total de registros: {df.count()}")

except Exception as e:
    # Si algo falla, mostramos el error sin que el programa colapse
    print(f" Error al leer el archivo: {str(e)}")

In [0]:
# Separar válidos e inválidos con Try/Except
from pyspark.sql.functions import col

try:
    # Registros válidos — donde Class es 0 o 1
    df_validos = df.filter(col("Class").isin([0, 1]))
    
    # Registros inválidos — donde algún campo importante es nulo
    df_invalidos = df.filter(
        col("Time").isNull() |
        col("Amount").isNull() |
        col("Class").isNull()
    )
    
    print(f"Registros válidos: {df_validos.count()}")
    print(f" Registros con errores: {df_invalidos.count()}")

except Exception as e:
    print(f" Error al separar los datos: {str(e)}")

In [0]:
# Guardar en Delta tables con Try/Except
try:
    # Guardamos los registros válidos
    df_validos.write.format("delta").mode("overwrite") \
        .saveAsTable("fraud_detection.bronze.transacciones_validas")
    
    # Guardamos los registros con errores
    df_invalidos.write.format("delta").mode("overwrite") \
        .saveAsTable("fraud_detection.bronze.transacciones_con_errores")
    
    print("Tablas guardadas correctamente")

except Exception as e:
    print(f"Error al guardar las tablas: {str(e)}")

In [0]:
# Ver los primeros registros
display(df_validos)